 # XGBoost for exceedance probability of hourly avg (>2 ppb)

## Load libraries

In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.model_selection import LeaveOneGroupOut, GroupKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
    mean_pinball_loss,
)
import joblib
from itertools import product
import time
from sklearn.model_selection import train_test_split

## Prepare data set

In [2]:
# Load dataset
save_path = "../rfiles/qxgb_files/"
file_path = "../rfiles/data/"
ha_full_full = pd.read_parquet(file_path + "ha_full_full_for_py_v2.parquet")
ha_full_full_y = pd.read_parquet(file_path + "ha_full_full_y_for_py_v2.parquet")
monitor_coord_map = pd.read_parquet(save_path +'monitor_coord_map.parquet')

In [3]:
hourly_predictors = ['year',
                     'month_sin', 'month_cos',
                     'hour_sin', 'hour_cos',
                     'weekday_sin', 'weekday_cos',
                     'julian_sin', 'julian_cos',
                     'wd_sin', 'wd_cos', 'ws_avg', 
                     'hourly_downwind_ref', 'dist_wrp', 'dist_ref',
                     'mon_utm_x', 'mon_utm_y', 'monthly_oil_2km', 'monthly_gas_2km', 
                     'active_2km', 'inactive_2km', 'hourly_downwind_wrp', 
                     'elevation', 'EVI', 'num_odor_complaints',
                     'dist_dc', 'closest_wrp_capacity', 'hourly_temp', 'hourly_hum', 'hourly_precip']

response = "H2S_hourly_avg"
time_col = "dayhour"
THRESH_PPB = 2.0
THRESH_LOG = np.log(THRESH_PPB)

df = ha_full_full.copy()

# site_id: use coords; if you have a Monitor/site column, use that instead
df["site_id"] = df[["mon_utm_x", "mon_utm_y"]].astype(str).agg("_".join, axis=1)
df = df.merge(monitor_coord_map, on=["mon_utm_x", "mon_utm_y"], how="left")
# Targets
df["y"] = ha_full_full_y[response]
df["y_exc"] = (df["y"] > THRESH_PPB).astype(int)
df["z"] = np.log(df["y"])

# Features
X_all = df[hourly_predictors]
y_raw = df["y"].to_numpy()
y_exc = df["y_exc"].to_numpy()
groups = df["site_id"].to_numpy()

In [4]:
df.head()

,year,month_sin,month_cos,hour_sin,hour_cos,weekday_sin,weekday_cos,julian_sin,julian_cos,wd_sin,...,hourly_temp,hourly_hum,hourly_precip,day,dayhour,site_id,Monitor,y,y_exc,z
0,2021.0,-0.866025,0.5,0.500000,-0.866025,-0.974928,-0.222521,-0.97486,0.22282,0.707107,...,63.4,47,0.0,2021-10-14 00:00:00-07:00,2021-10-14 10,383547.1749268095_3744771.0861730943,Chico,114.987778,1,4.744826
1,2021.0,-0.866025,0.5,0.258819,-0.965926,-0.974928,-0.222521,-0.97486,0.22282,0.224951,...,69.7,28,0.0,2021-10-14 00:00:00-07:00,2021-10-14 11,383547.1749268095_3744771.0861730943,Chico,131.718000,1,4.880663
2,2021.0,-0.866025,0.5,-0.500000,-0.866025,-0.974928,-0.222521,-0.97486,0.22282,-0.766044,...,78.3,21,0.0,2021-10-14 00:00:00-07:00,2021-10-14 14,383547.1749268095_3744771.0861730943,Chico,317.979167,1,5.761986
3,2021.0,-0.866025,0.5,-0.707107,-0.707107,-0.974928,-0.222521,-0.97486,0.22282,-0.866025,...,77.9,22,0.0,2021-10-14 00:00:00-07:00,2021-10-14 15,383547.1749268095_3744771.0861730943,Chico,269.660000,1,5.597162
4,2021.0,-0.866025,0.5,-0.866025,-0.500000,-0.974928,-0.222521,-0.97486,0.22282,-0.984808,...,75.1,29,0.0,2021-10-14 00:00:00-07:00,2021-10-14 16,383547.1749268095_3744771.0861730943,Chico,290.639167,1,5.672083


## Prepare helper functions

In [5]:
# Define a function that creates a closure to pass the quantile parameter
def pinball_loss_sklearn(preds, dtrain):
    y = dtrain.get_label()
    w = dtrain.get_weight()
    n = y.shape[0]
    K = len(alphas)
    # Handle both shapes returned by XGBoost: (n*K,) or (n, K)
    if preds.ndim == 1:
        if preds.size != n * K:
            raise ValueError(f"preds.size={preds.size} != n*K={n*K}")
        P = preds.reshape(n, K)
    else:
        if preds.shape != (n, K):
            raise ValueError(f"preds shape {preds.shape} != (n={n}, K={K})")
        P = preds
    # Compute pinball per quantile with its own alpha; average them
    losses = [
        mean_pinball_loss(y, P[:, j], alpha=alphas[j],
                          sample_weight=(w if w.size else None))
        for j in range(K)
    ]
    return "avg_pinball", float(np.mean(losses))

In [6]:
def build_xgb_params_from_best_row(best_row, *, objective, eval_metric, extra=None):
    """
    Map your tuned params into xgboost sklearn API params.
    - eta -> learning_rate
    - use max_leaves with grow_policy='lossguide' (best match for max_leaves tuning)
    """

    params = dict(
        max_leaves=int(best_row["max_leaves"]),
        learning_rate=float(best_row["eta"]),
        gamma=float(best_row["gamma"]),
        colsample_bytree=float(best_row["colsample_bytree"]),
        min_child_weight=float(best_row["min_child_weight"]),
        subsample=float(best_row["subsample"]),
        tree_method="hist",
        grow_policy="lossguide",
        objective=objective,
        random_state=42,
    )
    
    if "n_estimators" in best_row:
        params.update(dict(n_estimators = int(best_row["n_estimators"])))
    elif "mean_best_iteration" in best_row:
        params.update(dict(n_estimators = round(best_row["mean_best_iteration"])))

    if "reg_lambda" in best_row:
        params.update(dict(reg_lambda = int(best_row["reg_lambda"])))
                           
    if objective == "reg:quantileerror":
        params.update(dict(custom_metric = eval_metric))
    elif objective == "binary:logistic":
        params.update(dict(eval_metric = eval_metric))
        
    if extra:
        params.update(extra)
    return params

In [7]:
# ------------------------------------------------------------
#  Lag helpers
#  lag(t) = mean z across ALL training sites at time t
# ------------------------------------------------------------
def add_lag_columns_train_log(df_train, *, time_col, z_col="z"):
    """lag(t) = mean log(y) across ALL training sites at time t (same lag for rows at t)."""
    mean_by_time = df_train.groupby(time_col)[z_col].mean()
    out = df_train.copy()
    out["lag"] = out[time_col].map(mean_by_time)   # lag is on log scale
    return out

def add_lag_column_test_log(df_test, df_train, *, time_col, z_col="z"):
    """Apply lag(t) from training sites to held-out site rows (log scale)."""
    mean_by_time = df_train.groupby(time_col)[z_col].mean()
    out = df_test.copy()
    out["lag"] = out[time_col].map(mean_by_time)   # lag is on log scale
    return out

In [8]:
def _to_mat(pred, n, K):
    pred = np.asarray(pred)
    if pred.ndim == 1:
        if pred.size != n * K:
            raise ValueError(f"pred.size={pred.size} != n*K={n*K}")
        return pred.reshape(n, K)
    if pred.shape != (n, K):
        raise ValueError(f"pred shape {pred.shape} != (n={n}, K={K})")
    return pred

def monotone_in_p(q_mat):
    """Enforce nondecreasing quantiles in p (simple rearrangement via cumulative max)."""
    return np.maximum.accumulate(np.asarray(q_mat, dtype=float), axis=1)

def exceed_prob_from_quantiles(q_mat, ps, threshold):
    """
    Convert predicted quantiles Q(p) into P(Y > threshold) by inverting Q(p)=threshold.
    Piecewise-linear inversion in (p, q).
    """
    ps = np.asarray(ps, dtype=float)
    q = monotone_in_p(q_mat)
    n, K = q.shape
    thr = float(threshold)

    # insertion point: first index where q > thr
    idx = np.sum(q <= thr, axis=1)        # in [0..K]
    idx = np.clip(idx, 1, K - 1)          # force bracketing
    j0 = idx - 1
    j1 = idx

    q0 = q[np.arange(n), j0]
    q1 = q[np.arange(n), j1]
    p0 = ps[j0]
    p1 = ps[j1]

    denom = np.where(np.abs(q1 - q0) < 1e-12, 1.0, (q1 - q0))
    frac = np.clip((thr - q0) / denom, 0.0, 1.0)
    p_star = p0 + (p1 - p0) * frac   # approx F(threshold)
    exc = 1.0 - p_star

    # out-of-range handling
    below = thr <= q[:, 0]
    above = thr >= q[:, -1]
    exc[below] = 1.0 - ps[0]
    exc[above] = 1.0 - ps[-1]
    return np.clip(exc, 0.0, 1.0)

## New tuning (Same as ones used for 30 ppb)

### Binary classifier

In [93]:
tune_grid = pd.DataFrame(list(product(
    [64, 128, 256],               # max_leaves
    [0.05, 0.1, 0.2],               # eta
    [0.01, 0.1, 1],            # gamma
    [0.25, 0.5, 0.75],          # colsample_bytree
    [1.0, 5.0, 10],          # min_child_weight
    [0.25, 0.5, 0.75],         # subsample
    [5, 20, 50]    # L2
)), columns=[
    "max_leaves", "eta", "gamma", "colsample_bytree", "min_child_weight", "subsample", "reg_lambda"
])

In [94]:
tune_grid2 = tune_grid.sample(n=300, random_state=42).reset_index(drop=True)

In [95]:
# ============================================================
# 1) Make "leave 3 monitors out" folds (5 folds for 15 monitors)
#    Greedy-balance folds by exceedance count because exceedance is rare.
# ============================================================
def make_3_monitor_folds_balanced(df, group_col="Monitor", y_exc_col="y_exc", n_folds=5, seed=42):
    rng = np.random.default_rng(seed)

    exc = df.groupby(group_col)[y_exc_col].sum().sort_values(ascending=False)
    monitors = exc.index.to_list()

    # Shuffle within equal-count blocks to avoid deterministic ties
    ordered = []
    for c in sorted(exc.unique(), reverse=True):
        block = [m for m in monitors if exc[m] == c]
        rng.shuffle(block)
        ordered.extend(block)

    fold_monitors = [[] for _ in range(n_folds)]
    fold_exc = np.zeros(n_folds, dtype=float)

    for m in ordered:
        j = int(np.argmin(fold_exc))
        fold_monitors[j].append(m)
        fold_exc[j] += float(exc[m])

    folds = []
    all_idx = np.arange(len(df))
    for j in range(n_folds):
        test_mons = set(fold_monitors[j])
        test_mask = df[group_col].isin(test_mons).to_numpy()
        te = all_idx[test_mask]
        tr = all_idx[~test_mask]
        folds.append((tr, te))

    return folds, fold_monitors, fold_exc


# ============================================================
# 2) CV tuning with early stopping (NO LAG for tuning)
#    - Use n_estimators=2000
#    - Early stopping chooses best_iteration per fold
#    - Selection metric: mean PR-AUC (average precision) across folds
# ============================================================
def tune_xgb_exceedance_groupcv_earlystop(
    df,
    predictors,
    tune_grid,
    *,
    group_col="Monitor",
    y_exc_col="y_exc",
    n_folds=5,
    seed=42,
    n_estimators=2000,
    early_stopping_rounds=50,
    verbose_every=25,
):
    folds, fold_mons, fold_exc = make_3_monitor_folds_balanced(
        df, group_col=group_col, y_exc_col=y_exc_col, n_folds=n_folds, seed=seed
    )

    results = []
    for gi, row in tune_grid.iterrows():
        base_params = dict(
            n_estimators=int(n_estimators),
            max_leaves=int(row["max_leaves"]),
            learning_rate=float(row["eta"]),
            gamma=float(row["gamma"]),
            colsample_bytree=float(row["colsample_bytree"]),
            min_child_weight=float(row["min_child_weight"]),
            subsample=float(row["subsample"]),
            reg_lambda=float(row["reg_lambda"]),
            tree_method="hist",
            grow_policy="lossguide",
            max_depth=0,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1,
        )

        fold_ap, fold_ll, fold_bs, fold_best_iter = [], [], [], []
        fold_posrate, fold_n = [], []

        for (tr, te) in folds:
            dtr = df.iloc[tr]
            dte = df.iloc[te]

            Xtr = dtr[predictors]
            Xte = dte[predictors]
            ytr = dtr[y_exc_col].to_numpy(dtype=int)
            yte = dte[y_exc_col].to_numpy(dtype=int)

            # per-fold imbalance weight
            pos = int(ytr.sum())
            neg = int(len(ytr) - pos)
            spw = neg / max(pos, 1)

            clf = xgb.XGBClassifier(**base_params, scale_pos_weight=spw, early_stopping_rounds = early_stopping_rounds)

            # Early stopping uses the held-out 3-monitor fold as eval_set
            clf.fit(
                Xtr, ytr,
                eval_set=[(Xte, yte)],
                verbose=False
            )

            p = clf.predict_proba(Xte)[:, 1]

            fold_ap.append(float(average_precision_score(yte, p)))
            fold_ll.append(float(log_loss(yte, p, labels=[0, 1])))
            fold_bs.append(float(brier_score_loss(yte, p)))
            fold_best_iter.append(int(getattr(clf, "best_iteration", np.nan)))
            fold_posrate.append(float(yte.mean()))
            fold_n.append(int(len(yte)))

        out = dict(
            grid_id=int(gi),
            mean_pr_auc=float(np.mean(fold_ap)),
            mean_logloss=float(np.mean(fold_ll)),
            mean_brier=float(np.mean(fold_bs)),
            mean_best_iteration=float(np.nanmean(fold_best_iter)),
            std_pr_auc=float(np.std(fold_ap)),
            std_logloss=float(np.std(fold_ll)),
            std_brier=float(np.std(fold_bs)),
        )
        out.update({k: float(v) for k, v in row.to_dict().items()})
        results.append(out)

        if verbose_every and ((gi + 1) % verbose_every == 0):
            print(f"[{gi+1}/{len(tune_grid)}] PR-AUC={out['mean_pr_auc']:.6f} "
                  f"logloss={out['mean_logloss']:.6f} brier={out['mean_brier']:.6f} "
                  f"best_iter~{out['mean_best_iteration']:.1f}")

    results_df = pd.DataFrame(results)

    # Select best: lowest brier
    results_df = results_df.sort_values(
        ["mean_brier", "mean_logloss"],
        ascending=[True, True]
    ).reset_index(drop=True)

    best_row = results_df.iloc[0].to_dict()
    return best_row, results_df

In [97]:
# ============================================================
# 3) Run tuning
# ============================================================
start_time = time.perf_counter()
best_row, cv_results = tune_xgb_exceedance_groupcv_earlystop(
    df=df,
    predictors=hourly_predictors,
    tune_grid=tune_grid2,
    group_col="Monitor",
    y_exc_col="y_exc",
    n_folds=5,                 # 15 monitors -> 5 folds -> ~3 monitors held out each fold
    seed=42,
    n_estimators=2000,
    early_stopping_rounds=50,
    verbose_every=1,
)
end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"Execution time: {execution_time:.4f} seconds")

# Save results
cv_results.to_parquet(save_path +"bin_cv_results_2ppb.parquet", index=False)

[1/300] PR-AUC=0.502066 logloss=0.245932 brier=0.067928 best_iter~426.2
[2/300] PR-AUC=0.475729 logloss=0.270908 brier=0.078846 best_iter~250.4
[3/300] PR-AUC=0.502268 logloss=0.275900 brier=0.081169 best_iter~1023.4
[4/300] PR-AUC=0.507942 logloss=0.237431 brier=0.067652 best_iter~1078.2
[5/300] PR-AUC=0.495068 logloss=0.238846 brier=0.066325 best_iter~476.8
[6/300] PR-AUC=0.489928 logloss=0.245027 brier=0.066720 best_iter~141.6
[7/300] PR-AUC=0.496297 logloss=0.259246 brier=0.073283 best_iter~270.8
[8/300] PR-AUC=0.504342 logloss=0.250187 brier=0.068070 best_iter~353.4
[9/300] PR-AUC=0.496112 logloss=0.257993 brier=0.073372 best_iter~188.8
[10/300] PR-AUC=0.490082 logloss=0.284751 brier=0.082753 best_iter~205.4
[11/300] PR-AUC=0.514596 logloss=0.237822 brier=0.066387 best_iter~752.8
[12/300] PR-AUC=0.517947 logloss=0.220398 brier=0.059994 best_iter~1214.0
[13/300] PR-AUC=0.508582 logloss=0.222368 brier=0.060131 best_iter~480.8
[14/300] PR-AUC=0.478150 logloss=0.256772 brier=0.072272 

9672.2154 secs

In [98]:
bin_cv_results = pd.read_parquet(save_path +"bin_cv_results_2ppb.parquet")

best_row_bin = bin_cv_results.iloc[0].to_dict()
print("\nBest row (by mean brier, then logloss):")
print(best_row_bin)


Best row (by mean brier, then logloss):
{'grid_id': 190.0, 'mean_pr_auc': 0.5089342277146838, 'mean_logloss': 0.22287518463680733, 'mean_brier': 0.05983623940956421, 'mean_best_iteration': 672.0, 'std_pr_auc': 0.081621296551221, 'std_logloss': 0.033852841640598715, 'std_brier': 0.008989170335866025, 'max_leaves': 256.0, 'eta': 0.1, 'gamma': 0.01, 'colsample_bytree': 0.25, 'min_child_weight': 5.0, 'subsample': 0.75, 'reg_lambda': 50.0}


### QXGB

In [99]:
# ============================================================
# QXGB tuning with early stopping (NO lag for tuning)
# Scoring modes:
#   - "pinball": average pinball loss across quantiles (lower better)
#   - "exceed_logloss": logloss of implied exceedance prob at 30 (lower better)
#   - "exceed_brier": brier of implied exceedance prob at 30 (lower better)
#   - "exceed_pr_auc": PR-AUC of implied exceedance prob at 30 (higher better)
# ============================================================
def tune_qxgb_groupcv_earlystop(
    df,
    predictors,
    tune_grid,
    alphas,
    *,
    response_col="y",
    group_col="Monitor",
    y_exc_col="y_exc",
    n_estimators=2000,
    early_stopping_rounds=100,
    n_folds=5,
    seed=42,
    score_mode="pinball",
    verbose_every=25,
):
    folds, fold_mons, fold_exc = make_3_monitor_folds_balanced(
        df, group_col=group_col, y_exc_col=y_exc_col, n_folds=n_folds, seed=seed
    )

    results = []
    K = len(alphas)

    for gi, row in tune_grid.iterrows():
        base_params = dict(
            n_estimators=int(n_estimators),
            max_leaves=int(row["max_leaves"]),
            learning_rate=float(row["eta"]),
            gamma=float(row["gamma"]),
            colsample_bytree=float(row["colsample_bytree"]),
            min_child_weight=float(row["min_child_weight"]),
            subsample=float(row["subsample"]),
            reg_lambda=float(row.get("reg_lambda", 1.0)),
            tree_method="hist",
            grow_policy="lossguide",
            max_depth=0,
            objective="reg:quantileerror",
            random_state=42,
            n_jobs=-1,
        )

        fold_scores = []
        fold_best_iter = []

        for (tr, te) in folds:
            dtr = df.iloc[tr]
            dte = df.iloc[te]

            Xtr = dtr[predictors]
            Xte = dte[predictors]

            ytr_log = np.log(dtr[response_col].to_numpy(dtype=float))
            yte_log = np.log(dte[response_col].to_numpy(dtype=float))
            yte_exc = dte[y_exc_col].to_numpy(dtype=int)

            # Fit QXGB on log(y)
            qreg = xgb.XGBRegressor(
                **base_params,
                quantile_alpha=alphas,
                early_stopping_rounds=early_stopping_rounds
            )

            # Early stopping: use first alpha's built-in metric for stopping signal.
            # (Your selection metric is computed manually below.)
            qreg.fit(
                Xtr, ytr_log,
                eval_set=[(Xte, yte_log)],
                verbose=False
            )
            fold_best_iter.append(int(getattr(qreg, "best_iteration", np.nan)))

            # Predict quantiles of log(y)
            Qlog = _to_mat(qreg.predict(Xte), n=len(Xte), K=K)
            Qlog = monotone_in_p(Qlog)

            if score_mode == "pinball":
                # average pinball across quantiles on log scale
                losses = [
                    mean_pinball_loss(yte_log, Qlog[:, j], alpha=alphas[j])
                    for j in range(K)
                ]
                score = float(np.mean(losses))  # lower better

            else:
                # implied exceedance prob at 30 via inversion at log(30)
                p_exc = exceed_prob_from_quantiles(Qlog, ps=alphas, threshold=THRESH_LOG)

                if score_mode == "exceed_logloss":
                    score = float(log_loss(yte_exc, p_exc, labels=[0, 1]))  # lower better
                elif score_mode == "exceed_brier":
                    score = float(brier_score_loss(yte_exc, p_exc))         # lower better
                elif score_mode == "exceed_pr_auc":
                    score = float(average_precision_score(yte_exc, p_exc))  # higher better
                else:
                    raise ValueError(f"Unknown score_mode: {score_mode}")

            fold_scores.append(score)

        # Aggregate across folds
        mean_score = float(np.mean(fold_scores))
        std_score = float(np.std(fold_scores))
        mean_best_iter = float(np.nanmean(fold_best_iter))

        out = dict(
            grid_id=int(gi),
            score_mode=score_mode,
            mean_score=mean_score,
            std_score=std_score,
            mean_best_iteration=mean_best_iter,
        )
        out.update({k: float(v) for k, v in row.to_dict().items()})
        results.append(out)

        if verbose_every and ((gi + 1) % verbose_every == 0):
            print(f"[{gi+1}/{len(tune_grid)}] mode={score_mode} mean_score={mean_score:.6f} "
                  f"best_iter~{mean_best_iter:.1f}")

    results_df = pd.DataFrame(results)

    # Choose best depending on score_mode direction
    if score_mode == "exceed_pr_auc":
        results_df = results_df.sort_values(["mean_score", "std_score"], ascending=[False, True]).reset_index(drop=True)
    else:
        results_df = results_df.sort_values(["mean_score", "std_score"], ascending=[True, True]).reset_index(drop=True)

    best_row = results_df.iloc[0].to_dict()
    return best_row, results_df

In [100]:
tune_alphas = [0.01, 0.1, 0.5, 0.9, 0.99]
tune_alphas

[0.01, 0.1, 0.5, 0.9, 0.99]

In [101]:
tune_grid3 = tune_grid.sample(n=100, random_state=42).reset_index(drop=True)

In [102]:
# ============================================================
# Run tuning (pick one score mode)
# ============================================================
start_time = time.perf_counter()
best_row_q, qxgb_results = tune_qxgb_groupcv_earlystop(
    df=df,
    predictors=hourly_predictors,
    tune_grid=tune_grid3,
    alphas=tune_alphas,
    response_col="y",
    group_col="Monitor",
    y_exc_col="y_exc",
    n_estimators=2000,
    early_stopping_rounds=25,
    n_folds=5,          
    seed=42,
    score_mode="exceed_brier",   # or "pinball", "exceed_brier", "exceed_pr_auc"
    verbose_every=1,
)
end_time = time.perf_counter()

execution_time = end_time - start_time
print(f"Execution time: {execution_time:.4f} seconds")

print("\nBest QXGB row:")
print(best_row_q)

# Save results
qxgb_results.to_parquet(save_path +"qxgb_cv_results_2ppb.parquet", index=False)

[1/100] mode=exceed_brier mean_score=0.058683 best_iter~34.2
[2/100] mode=exceed_brier mean_score=0.066950 best_iter~13.2
[3/100] mode=exceed_brier mean_score=0.059840 best_iter~61.6
[4/100] mode=exceed_brier mean_score=0.064480 best_iter~65.8
[5/100] mode=exceed_brier mean_score=0.065130 best_iter~27.4
[6/100] mode=exceed_brier mean_score=0.060547 best_iter~17.2
[7/100] mode=exceed_brier mean_score=0.061162 best_iter~27.0
[8/100] mode=exceed_brier mean_score=0.060685 best_iter~14.0
[9/100] mode=exceed_brier mean_score=0.059354 best_iter~19.4
[10/100] mode=exceed_brier mean_score=0.059584 best_iter~29.2
[11/100] mode=exceed_brier mean_score=0.057753 best_iter~88.4
[12/100] mode=exceed_brier mean_score=0.065679 best_iter~53.2
[13/100] mode=exceed_brier mean_score=0.065127 best_iter~30.4
[14/100] mode=exceed_brier mean_score=0.063270 best_iter~39.0
[15/100] mode=exceed_brier mean_score=0.059275 best_iter~64.6
[16/100] mode=exceed_brier mean_score=0.059048 best_iter~75.2
[17/100] mode=exc

4071.1942 secs

In [103]:
qxgb_cv_results = pd.read_parquet(save_path +"qxgb_cv_results_2ppb.parquet")

best_row_qxgb = qxgb_cv_results.iloc[0].to_dict()
print("\nBest row (by mean brier, then std):")
print(best_row_qxgb)


Best row (by mean brier, then std):
{'grid_id': 50, 'score_mode': 'exceed_brier', 'mean_score': 0.057745776579423126, 'std_score': 0.009788955477303284, 'mean_best_iteration': 38.8, 'max_leaves': 128.0, 'eta': 0.1, 'gamma': 1.0, 'colsample_bytree': 0.75, 'min_child_weight': 1.0, 'subsample': 0.75, 'reg_lambda': 5.0}


### Set new params

In [104]:
# params for binary exceedance model
bin_params_new = build_xgb_params_from_best_row(
    best_row_bin,
    objective="binary:logistic",
    eval_metric="logloss"
)

# params for quantile model (pinball loss)
# NOTE: objective will be set in the regressor constructor; eval_metric can be omitted or set.
q_params_new = build_xgb_params_from_best_row(
    best_row_qxgb,
    objective="reg:quantileerror",
    eval_metric=pinball_loss_sklearn
)

In [105]:
bin_params_new

{'max_leaves': 256,
 'learning_rate': 0.1,
 'gamma': 0.01,
 'colsample_bytree': 0.25,
 'min_child_weight': 5.0,
 'subsample': 0.75,
 'tree_method': 'hist',
 'grow_policy': 'lossguide',
 'objective': 'binary:logistic',
 'random_state': 42,
 'n_estimators': 672,
 'reg_lambda': 50,
 'eval_metric': 'logloss'}

In [106]:
q_params_new

{'max_leaves': 128,
 'learning_rate': 0.1,
 'gamma': 1.0,
 'colsample_bytree': 0.75,
 'min_child_weight': 1.0,
 'subsample': 0.75,
 'tree_method': 'hist',
 'grow_policy': 'lossguide',
 'objective': 'reg:quantileerror',
 'random_state': 42,
 'n_estimators': 39,
 'reg_lambda': 5,
 'custom_metric': <function __main__.pinball_loss_sklearn(preds, dtrain)>}

## Spatial test

### Helper functions

In [107]:
loso = LeaveOneGroupOut()

def loso_binary_with_lag_log(df, predictors, loso, bin_params, *, time_col, verbose=True):
    metrics_rows = []
    pred_parts = []

    for fold, (tr, te) in enumerate(
        loso.split(df, df["y_exc"].to_numpy(), groups=df["site_id"].to_numpy()),
        start=1
    ):
        train = df.iloc[tr].copy()
        test  = df.iloc[te].copy()
        site_id = str(test["site_id"].iloc[0])

        # LOG-scale lag
        train = add_lag_columns_train_log(train, time_col=time_col, z_col="z")
        test  = add_lag_column_test_log(test, train, time_col=time_col, z_col="z")

        train = train.dropna(subset=["lag"])
        test  = test.dropna(subset=["lag"])

        Xtr = train[predictors + ["lag"]]
        ytr = train["y_exc"].to_numpy()
        Xte = test[predictors + ["lag"]]
        yte = test["y_exc"].to_numpy()

        # imbalance weight
        pos = max(int(ytr.sum()), 1)
        neg = max(int(len(ytr) - ytr.sum()), 1)
        scale_pos_weight = neg / pos

        clf = xgb.XGBClassifier(**bin_params, scale_pos_weight=scale_pos_weight)
        clf.fit(Xtr, ytr)
        p = clf.predict_proba(Xte)[:, 1]

        # ---- store per-row predictions for this fold ----
        pred_fold = test[[time_col, "mon_utm_x", "mon_utm_y", "Monitor"]].copy()
        pred_fold["obs"] = test["y"].to_numpy()  # observed ppb
        pred_fold["pred_exceedance_probability"] = p
        pred_fold["fold"] = fold
        pred_fold["site"] = site_id
        pred_parts.append(pred_fold)

        # ---- fold metrics ----
        row = dict(
            fold=fold, site=site_id,
            n_test=len(yte),
            pos_rate_test=float(yte.mean()),
            brier=brier_score_loss(yte, p),
            logloss=log_loss(yte, p, labels=[0, 1]),
        )
        if len(np.unique(yte)) == 2:
            row["roc_auc"] = roc_auc_score(yte, p)
            row["pr_auc"]  = average_precision_score(yte, p)
        else:
            row["roc_auc"] = np.nan
            row["pr_auc"]  = np.nan

        metrics_rows.append(row)

        if verbose:
            print(
                f"[LOSO Binary(log-lag)] fold={fold:02d} site={site_id} "
                f"n_test={row['n_test']} pos_rate={row['pos_rate_test']:.4f} "
                f"brier={row['brier']:.5f} logloss={row['logloss']:.5f} "
                f"roc_auc={row['roc_auc'] if not np.isnan(row['roc_auc']) else 'NA'} "
                f"pr_auc={row['pr_auc'] if not np.isnan(row['pr_auc']) else 'NA'}"
            )

    df_metrics = pd.DataFrame(metrics_rows)
    df_pred = pd.concat(pred_parts, ignore_index=True) if pred_parts else pd.DataFrame()

    return df_metrics, df_pred


def loso_quantile_residual_log(df, predictors, alphas, loso, q_params,
                               *, time_col, verbose=True):
    metrics_rows = []
    pred_parts = []

    for fold, (tr, te) in enumerate(
        loso.split(df, df["z"].to_numpy(), groups=df["site_id"].to_numpy()),
        start=1
    ):
        train = df.iloc[tr].copy()
        test  = df.iloc[te].copy()
        site_id = str(test["site_id"].iloc[0])

        # LOG-scale lag
        train = add_lag_columns_train_log(train, time_col=time_col, z_col="z")
        test  = add_lag_column_test_log(test, train, time_col=time_col, z_col="z")

        train = train.dropna(subset=["lag"]).copy()
        test  = test.dropna(subset=["lag"]).copy()

        # residual on log scale: r = log(y) - lag
        train["r"] = train["z"] - train["lag"]
        test["r"]  = test["z"]  - test["lag"]

        Xtr = train[predictors + ["lag"]]
        ytr = train["r"].to_numpy()
        Xte = test[predictors + ["lag"]]
        yte_r   = test["r"].to_numpy()
        yte_exc = test["y_exc"].to_numpy()

        qreg = xgb.XGBRegressor(
            **q_params,
            quantile_alpha=alphas,
        )
        qreg.fit(Xtr, ytr)

        # Residual quantiles on log-residual scale
        Qr = _to_mat(qreg.predict(Xte), n=len(Xte), K=len(alphas))

        # Reconstruct quantiles of Z = log(Y): Qz(p) = lag + Qr(p)
        Qz = Qr + test["lag"].to_numpy().reshape(-1, 1)

        # Pinball on residuals (log scale)
        pinballs = [mean_pinball_loss(yte_r, Qr[:, j], alpha=alphas[j]) for j in range(len(alphas))]
        avg_pinball = float(np.mean(pinballs))

        # Exceedance for Y>30 is Z>log(30)
        p_exc = exceed_prob_from_quantiles(Qz, ps=alphas, threshold=THRESH_LOG)

        # ---- store per-row predictions for this fold ----
        pred_fold = test[[time_col, "mon_utm_x", "mon_utm_y", "Monitor"]].copy()
        pred_fold["obs"] = test["y"].to_numpy()  # observed ppb
        pred_fold["pred_exceedance_probability"] = p_exc
        pred_fold["fold"] = fold
        pred_fold["site"] = site_id
        pred_parts.append(pred_fold)

        # ---- fold metrics ----
        row = dict(
            fold=fold, site=site_id,
            n_test=len(yte_exc),
            pos_rate_test=float(yte_exc.mean()),
            avg_pinball_resid=avg_pinball,
            brier_exc=brier_score_loss(yte_exc, p_exc),
            logloss_exc=log_loss(yte_exc, p_exc, labels=[0, 1]),
        )
        if len(np.unique(yte_exc)) == 2:
            row["roc_auc_exc"] = roc_auc_score(yte_exc, p_exc)
            row["pr_auc_exc"]  = average_precision_score(yte_exc, p_exc)
        else:
            row["roc_auc_exc"] = np.nan
            row["pr_auc_exc"]  = np.nan

        metrics_rows.append(row)

        if verbose:
            print(
                f"[LOSO Quantile(log-resid)] fold={fold:02d} site={site_id} "
                f"n_test={row['n_test']} pos_rate={row['pos_rate_test']:.4f} "
                f"avg_pinball_resid={row['avg_pinball_resid']:.5f} "
                f"brier_exc={row['brier_exc']:.5f} logloss_exc={row['logloss_exc']:.5f} "
                f"roc_auc_exc={row['roc_auc_exc'] if not np.isnan(row['roc_auc_exc']) else 'NA'} "
                f"pr_auc_exc={row['pr_auc_exc'] if not np.isnan(row['pr_auc_exc']) else 'NA'}"
            )

    df_metrics = pd.DataFrame(metrics_rows)
    df_pred = pd.concat(pred_parts, ignore_index=True) if pred_parts else pd.DataFrame()

    return df_metrics, df_pred

In [108]:
def _make_site_key(df, x_col="mon_utm_x", y_col="mon_utm_y"):
    return df[[x_col, y_col]].astype(str).agg("_".join, axis=1)

def make_monitor_loso_summary(
    monitor_df, df_bin_lag, df_q_resid,
    monitor_name_col="Monitor",
    x_col="mon_utm_x", y_col="mon_utm_y",
    metrics=None,                 # <- choose which metric columns to return
    include_checks=True           # <- include n_match/prop_match
):
    """
    Build a per-monitor LOSO summary and add:
      - baseline_brier = r(1-r) using prop_exceed (r)
      - BSS for each model:
            BSS = 1 - (Brier_model / baseline_brier)
        (computed for binary and quantile exceedance probs)
    Parameters
    ----------
    metrics : None or list[str]
        If None, returns a sensible default set of columns.
        If list, returns the requested metrics (plus identifiers + core counts).
        Allowed metric names (strings):
          "bin_brier","bin_logloss","bin_roc_auc","bin_pr_auc",
          "q_avg_pinball_resid","q_brier","q_logloss","q_roc_auc","q_pr_auc",
          "baseline_brier","bin_bss","q_bss",
          "n_match","prop_match"
    """

    # --- monitor lookup with site key ---
    mon = monitor_df.copy()
    mon["site"] = _make_site_key(mon, x_col=x_col, y_col=y_col)

    # --- binary LOSO metrics (one row per site/fold) ---
    bin_df = df_bin_lag.copy()
    bin_df["n_exceed"] = np.round(bin_df["pos_rate_test"] * bin_df["n_test"]).astype(int)

    bin_keep = bin_df[[
        "site", "n_test", "n_exceed", "pos_rate_test",
        "brier", "logloss", "roc_auc", "pr_auc"
    ]].rename(columns={
        "n_test": "n",
        "pos_rate_test": "prop_exceed",
        "brier": "bin_brier",
        "logloss": "bin_logloss",
        "roc_auc": "bin_roc_auc",
        "pr_auc": "bin_pr_auc",
    })

    # --- quantile LOSO metrics (one row per site/fold) ---
    q_df = df_q_resid.copy()
    q_df["n_exceed_q"] = np.round(q_df["pos_rate_test"] * q_df["n_test"]).astype(int)

    q_keep = q_df[[
        "site",
        "avg_pinball_resid", "brier_exc", "logloss_exc", "roc_auc_exc", "pr_auc_exc",
        "n_test", "pos_rate_test", "n_exceed_q"
    ]].rename(columns={
        "avg_pinball_resid": "q_avg_pinball_resid",
        "brier_exc": "q_brier",
        "logloss_exc": "q_logloss",
        "roc_auc_exc": "q_roc_auc",
        "pr_auc_exc": "q_pr_auc",
        "n_test": "n_q",
        "pos_rate_test": "prop_exceed_q",
        "n_exceed_q": "n_exceed_q",
    })

    # --- merge everything ---
    out = (
        mon.merge(bin_keep, on="site", how="left")
           .merge(q_keep, on="site", how="left")
    )

    # --- baseline Brier using the observed exceedance rate r = prop_exceed ---
    r = out["prop_exceed"].astype(float)
    out["baseline_brier"] = r * (1.0 - r)

    # --- Brier Skill Score (BSS) for each model ---
    # BSS = 1 - (Brier_model / baseline_brier)
    # handle baseline_brier==0 (all 0 or all 1): set BSS to NaN
    denom = out["baseline_brier"].to_numpy()
    denom_safe = np.where((denom > 0) & np.isfinite(denom), denom, np.nan)

    out["bin_bss"] = 1.0 - (out["bin_brier"].to_numpy() / denom_safe)
    out["q_bss"]   = 1.0 - (out["q_brier"].to_numpy() / denom_safe)

    # Optional: show whether model beats baseline directly
    out["bin_brier_better_than_baseline"] = out["bin_brier"] < out["baseline_brier"]
    out["q_brier_better_than_baseline"]   = out["q_brier"] < out["baseline_brier"]

    # --- Optional consistency checks ---
    if include_checks:
        out["n_match"] = (out["n"].astype("float") == out["n_q"].astype("float"))
        out["prop_match"] = np.isclose(
            out["prop_exceed"].astype("float"),
            out["prop_exceed_q"].astype("float"),
            equal_nan=True
        )

    # --- default columns / user-selected metric columns ---
    id_cols = [monitor_name_col]
    core_cols = ["n", "n_exceed", "prop_exceed", "baseline_brier"]

    default_metrics = [
        "bin_brier", "bin_bss", "bin_logloss", "bin_roc_auc", "bin_pr_auc",
        "q_brier", "q_bss", "q_logloss", "q_roc_auc", "q_pr_auc",
        "q_avg_pinball_resid",
        "bin_brier_better_than_baseline", "q_brier_better_than_baseline",
    ]
    if include_checks:
        default_metrics += ["n_match", "prop_match"]

    if metrics is None:
        metrics = default_metrics

    # Ensure only existing columns are requested
    wanted = id_cols + core_cols + metrics
    wanted = [c for c in wanted if c in out.columns]
    out = out[wanted]

    # Warn if any monitors didn't match a fold (site key mismatch)
    missing = out[out["n"].isna() & out.get("q_brier", pd.Series(np.nan, index=out.index)).isna()]
    if len(missing) > 0:
        print("WARNING: Some monitors did not match any LOSO fold 'site' key (check site_id construction).")
        print(missing[[monitor_name_col, x_col, y_col]].head(20))

    return out.sort_values(monitor_name_col)

In [109]:
def count_in_alpha_intervals(df, col, alphas):
    """
    Given alphas (prob levels) and a numeric column in df, count how many values
    fall into each interval formed by the alphas.

    Intervals are:
      (-inf, a1], (a1, a2], ..., (a_{K-1}, aK], (aK, +inf)

    Returns a DataFrame with interval labels and counts.
    """
    alphas = np.asarray(sorted(set(alphas)), dtype=float)
    x = df[col].to_numpy(dtype=float)
    x = 1 - x
    
    bins = np.concatenate(([-np.inf], alphas, [np.inf]))

    labels = []
    labels.append(f"(-inf, {alphas[0]}]")
    for i in range(len(alphas) - 1):
        labels.append(f"({alphas[i]}, {alphas[i+1]}]")
    labels.append(f"({alphas[-1]}, inf)")

    cats = pd.cut(x, bins=bins, right=True, include_lowest=True, labels=labels)

    # value_counts without sort kwarg, then reindex to preserve interval order
    vc = cats.value_counts(dropna=False)
    vc = vc.reindex(labels, fill_value=0)

    out = vc.reset_index()
    out.columns = ["interval", "count"]
    return out

In [110]:
def expected_exceedances_by_group(df_pred, *,
                                   group_col="Monitor",
                                   obs_col="obs",
                                   p_col="pred_exceedance_probability"):
    """
    Group by monitor, compute:
      - n
      - avg_pred_p
      - expected_exceed
      - observed_exceed
      - observed_rate
    """
    df = df_pred.copy()
    df["_exc_obs"] = (df[obs_col].to_numpy(dtype=float) > float(THRESH_PPB)).astype(int)
    
    out = (
        df.groupby(group_col, as_index=False)
          .agg(
              n=(obs_col, "size"),
              observed_exceed=("_exc_obs", "sum"),
              expected_exceed=(p_col, lambda s: float(s.mean()) * s.size),
          )
    )
    out["expected_exceed"] = np.rint(out["expected_exceed"]).astype(int)

    # Overall row (pooled over all observations)
    n_group = np.rint(out["n"].mean()).astype(int)              

    mean = pd.DataFrame([{
        group_col: "Mean",
        "n": np.rint(out["n"].mean()).astype(int)  ,
        "observed_exceed": np.rint(out["observed_exceed"].mean()).astype(int)    ,
        "expected_exceed": np.rint(out["expected_exceed"].mean()).astype(int)       
    }])

    sums = pd.DataFrame([{
        group_col: "Sum",
        "n": np.rint(out["n"].sum()).astype(int)  ,
        "observed_exceed": np.rint(out["observed_exceed"].sum()).astype(int)    ,
        "expected_exceed": np.rint(out["expected_exceed"].sum()).astype(int)       
    }])

    out = pd.concat([out, mean, sums], ignore_index=True)
    return out

### New tunings

In [111]:
file_suffix = "_2ppb_newtuning"

In [112]:
df_bin_lag_metrics_newtuning, df_bin_lag_pred_newtuning = loso_binary_with_lag_log(
    df, hourly_predictors, loso, bin_params_new, time_col=time_col, verbose=True
)

print("\n=== LOSO | Binary logistic (log-lag) | fold metrics ===")
print(df_bin_lag_metrics_newtuning.mean(numeric_only=True).round(4))

[LOSO Binary(log-lag)] fold=01 site=369852.5359163884_3754053.0239289817 n_test=29467 pos_rate=0.0342 brier=0.03349 logloss=0.14475 roc_auc=0.759765844988105 pr_auc=0.15384696363068134
[LOSO Binary(log-lag)] fold=02 site=369932.10194513993_3754245.360266213 n_test=5279 pos_rate=0.0591 brier=0.07092 logloss=0.45332 roc_auc=0.31026699292251936 pr_auc=0.039346431540771
[LOSO Binary(log-lag)] fold=03 site=370408.9455206478_3750865.96396696 n_test=30121 pos_rate=0.0198 brier=0.02733 logloss=0.10010 roc_auc=0.954608272896641 pr_auc=0.3674586893122522
[LOSO Binary(log-lag)] fold=04 site=373408.60980845685_3746402.1236518873 n_test=20070 pos_rate=0.0295 brier=0.18570 logloss=0.63119 roc_auc=0.7206583698280005 pr_auc=0.25671630463833844
[LOSO Binary(log-lag)] fold=05 site=376372.05412800366_3748307.7286123442 n_test=20257 pos_rate=0.0814 brier=0.05982 logloss=0.22469 roc_auc=0.864895747412125 pr_auc=0.4717603090206746
[LOSO Binary(log-lag)] fold=06 site=376793.2331856196_3745266.662266282 n_tes

In [113]:
df_bin_lag_metrics_newtuning.to_parquet(save_path +f"loso_binary_metrics{file_suffix}.parquet", index=False)
df_bin_lag_pred_newtuning.to_parquet(save_path +f"loso_binary_predictions{file_suffix}.parquet", index=False)

In [114]:
alphas = [0.001, 0.01, 0.03, 0.05, 0.075, 
          0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9,
          0.925, 0.95, 0.97, 0.99, 0.999, 0.9999, 0.99999, 0.999999, 0.9999999, 0.99999999, 0.999999999, 0.9999999999]

df_q_resid_metrics_newtuning, df_q_resid_pred_newtuning = loso_quantile_residual_log(
    df, hourly_predictors, alphas, loso, q_params_new, time_col=time_col, verbose=True
)

print("\n=== LOSO | Quantile residual (log-resid) | fold metrics ===")
print(df_q_resid_metrics_newtuning .mean(numeric_only=True).round(4))

[LOSO Quantile(log-resid)] fold=01 site=369852.5359163884_3754053.0239289817 n_test=29467 pos_rate=0.0342 avg_pinball_resid=0.15084 brier_exc=0.03367 logloss_exc=0.13376 roc_auc_exc=0.8157416356760583 pr_auc_exc=0.16612339056800873
[LOSO Quantile(log-resid)] fold=02 site=369932.10194513993_3754245.360266213 n_test=5279 pos_rate=0.0591 avg_pinball_resid=0.15892 brier_exc=0.06024 logloss_exc=0.38477 roc_auc_exc=0.2595005239710293 pr_auc_exc=0.03675041115293607
[LOSO Quantile(log-resid)] fold=03 site=370408.9455206478_3750865.96396696 n_test=30121 pos_rate=0.0198 avg_pinball_resid=0.12923 brier_exc=0.01529 logloss_exc=0.06215 roc_auc_exc=0.9700053278779507 pr_auc_exc=0.44763036455451655
[LOSO Quantile(log-resid)] fold=04 site=373408.60980845685_3746402.1236518873 n_test=20070 pos_rate=0.0295 avg_pinball_resid=0.23639 brier_exc=0.18855 logloss_exc=0.58589 roc_auc_exc=0.6679944459937657 pr_auc_exc=0.204913666209579
[LOSO Quantile(log-resid)] fold=05 site=376372.05412800366_3748307.728612344

In [115]:
df_q_resid_metrics_newtuning.to_parquet(save_path +f"loso_quantile_metrics{file_suffix}.parquet", index=False)
df_q_resid_pred_newtuning.to_parquet(save_path +f"loso_quantile_predictions{file_suffix}.parquet", index=False)

In [116]:
df_bin_metrics_newtuning = pd.read_parquet(save_path +f"loso_binary_metrics{file_suffix}.parquet")
df_bin_pred_newtuning    = pd.read_parquet(save_path +f"loso_binary_predictions{file_suffix}.parquet")

df_q_metrics_newtuning   = pd.read_parquet(save_path +f"loso_quantile_metrics{file_suffix}.parquet")
df_q_pred_newtuning      = pd.read_parquet(save_path +f"loso_quantile_predictions{file_suffix}.parquet")

In [117]:
# the interval represents the predicted quantiles that contain 30ppb
# 1-interval is the interval of exceedance probability
# so 249272 observations have 0.0000001*100% to 0.000001*100% exceedance probability
count_in_alpha_intervals(df_q_pred_newtuning, "pred_exceedance_probability", alphas)

,interval,count
0,"(-inf, 0.001]",0
1,"(0.001, 0.01]",508
2,"(0.01, 0.03]",1767
3,"(0.03, 0.05]",1196
4,"(0.05, 0.075]",1065
5,"(0.075, 0.1]",1845
6,"(0.1, 0.2]",4639
7,"(0.2, 0.3]",5902
8,"(0.3, 0.4]",7934
9,"(0.4, 0.5]",7068


In [118]:
make_monitor_loso_summary(monitor_coord_map, df_bin_metrics_newtuning, df_q_metrics_newtuning,
                          metrics = ["bin_brier", "bin_bss", "bin_roc_auc", "bin_pr_auc",
                                     "q_brier",  "q_bss", "q_roc_auc", "q_pr_auc"])

,Monitor,n,n_exceed,prop_exceed,baseline_brier,bin_brier,bin_bss,bin_roc_auc,bin_pr_auc,q_brier,q_bss,q_roc_auc,q_pr_auc
0,Chico,2521,2487,0.986513,0.013305,0.437374,-31.873362,0.766835,0.996266,0.566906,-41.609057,0.785177,0.996578
1,ElSegundo,5279,312,0.059102,0.055609,0.070917,-0.275279,0.310267,0.039346,0.060242,-0.083316,0.259501,0.036750
2,ElmAve,32216,2107,0.065402,0.061125,0.073272,-0.198723,0.919585,0.554275,0.045653,0.253125,0.880086,0.464847
3,FirstMethodist,30759,1935,0.062908,0.058951,0.045021,0.236290,0.937757,0.650301,0.036437,0.381918,0.938973,0.643052
4,GStreet,22217,1677,0.075483,0.069785,0.073654,-0.055443,0.892047,0.582420,0.052632,0.245805,0.870444,0.544556
5,GuenserPark,27326,1279,0.046805,0.044615,0.055801,-0.250728,0.920007,0.539098,0.030095,0.325446,0.945228,0.541181
6,HarborPark,26127,766,0.029318,0.028459,0.103585,-2.639819,0.839585,0.287414,0.063806,-1.242051,0.846113,0.307350
7,Hudson,32091,3559,0.110903,0.098604,0.080102,0.187634,0.946308,0.748238,0.052528,0.467279,0.941645,0.728394
8,InnerPort,29107,3905,0.134160,0.116161,0.116178,-0.000146,0.811085,0.532314,0.218611,-0.881958,0.769896,0.514036
9,Judson,30014,2467,0.082195,0.075439,0.070949,0.059524,0.967998,0.739741,0.042898,0.431351,0.961621,0.730325


For binary classifier:
- Model is useful for 5 sites (BSS > 0)
 
For QXGB:
- Model is useful for 9 sites

In [119]:
exp_bin = expected_exceedances_by_group(df_bin_pred_newtuning, group_col = "Monitor")
exp_q   = expected_exceedances_by_group(df_q_pred_newtuning, group_col = "Monitor")
exp_bin.merge(exp_q, on = ['Monitor', 'n'], how = 'left', suffixes = ('_bin', '_q'))

,Monitor,n,observed_exceed_bin,expected_exceed_bin,observed_exceed_q,expected_exceed_q
0,Chico,2521,2487,1145,2487,778
1,ElSegundo,5279,312,206,312,71
2,ElmAve,32216,2107,5669,2107,2142
3,FirstMethodist,30759,1935,3487,1935,1479
4,GStreet,22217,1677,3700,1677,2362
5,GuenserPark,27326,1279,3187,1279,1792
6,HarborPark,26127,766,4951,766,3277
7,Hudson,32091,3559,7112,3559,3260
8,InnerPort,29107,3905,6318,3905,12506
9,Judson,30014,2467,5693,2467,3387


In [120]:
file_suffix = "_2ppb_newtuning_wojulian"

In [121]:
hourly_wojulian_predictors = [p for p in hourly_predictors if p not in ['julian_cos', 'julian_sin']]
hourly_wojulian_predictors

['year',
 'month_sin',
 'month_cos',
 'hour_sin',
 'hour_cos',
 'weekday_sin',
 'weekday_cos',
 'wd_sin',
 'wd_cos',
 'ws_avg',
 'hourly_downwind_ref',
 'dist_wrp',
 'dist_ref',
 'mon_utm_x',
 'mon_utm_y',
 'monthly_oil_2km',
 'monthly_gas_2km',
 'active_2km',
 'inactive_2km',
 'hourly_downwind_wrp',
 'elevation',
 'EVI',
 'num_odor_complaints',
 'dist_dc',
 'closest_wrp_capacity',
 'hourly_temp',
 'hourly_hum',
 'hourly_precip']

In [122]:
df_bin_lag_metrics_newtuning_wojulian, df_bin_lag_pred_newtuning_wojulian = loso_binary_with_lag_log(
    df, hourly_wojulian_predictors, loso, bin_params_new, time_col=time_col, verbose=True
)

print("\n=== LOSO | Binary logistic (log-lag) | fold metrics ===")
print(df_bin_lag_metrics_newtuning_wojulian.mean(numeric_only=True).round(4))

[LOSO Binary(log-lag)] fold=01 site=369852.5359163884_3754053.0239289817 n_test=29467 pos_rate=0.0342 brier=0.03386 logloss=0.15372 roc_auc=0.7184471175419289 pr_auc=0.11582840194213968
[LOSO Binary(log-lag)] fold=02 site=369932.10194513993_3754245.360266213 n_test=5279 pos_rate=0.0591 brier=0.07054 logloss=0.46179 roc_auc=0.3039786952863257 pr_auc=0.03877071328143194
[LOSO Binary(log-lag)] fold=03 site=370408.9455206478_3750865.96396696 n_test=30121 pos_rate=0.0198 brier=0.02957 logloss=0.10883 roc_auc=0.9595746406670776 pr_auc=0.4072924561865932
[LOSO Binary(log-lag)] fold=04 site=373408.60980845685_3746402.1236518873 n_test=20070 pos_rate=0.0295 brier=0.19296 logloss=0.65939 roc_auc=0.7230124241322038 pr_auc=0.20322866988786456
[LOSO Binary(log-lag)] fold=05 site=376372.05412800366_3748307.7286123442 n_test=20257 pos_rate=0.0814 brier=0.06316 logloss=0.23181 roc_auc=0.8559636753173508 pr_auc=0.44533949559381986
[LOSO Binary(log-lag)] fold=06 site=376793.2331856196_3745266.662266282 

In [123]:
df_bin_lag_metrics_newtuning_wojulian.to_parquet(save_path +f"loso_binary_metrics{file_suffix}.parquet", index=False)
df_bin_lag_pred_newtuning_wojulian.to_parquet(save_path +f"loso_binary_predictions{file_suffix}.parquet", index=False)

In [124]:
df_q_resid_metrics_newtuning_wojulian, df_q_resid_pred_newtuning_wojulian = loso_quantile_residual_log(
    df, hourly_wojulian_predictors, alphas, loso, q_params_new, time_col=time_col, verbose=True
)

print("\n=== LOSO | Quantile residual (log-resid) | fold metrics ===")
print(df_q_resid_metrics_newtuning_wojulian.mean(numeric_only=True).round(4))

[LOSO Quantile(log-resid)] fold=01 site=369852.5359163884_3754053.0239289817 n_test=29467 pos_rate=0.0342 avg_pinball_resid=0.14297 brier_exc=0.03312 logloss_exc=0.13315 roc_auc_exc=0.8096333047445115 pr_auc_exc=0.17619631884552006
[LOSO Quantile(log-resid)] fold=02 site=369932.10194513993_3754245.360266213 n_test=5279 pos_rate=0.0591 avg_pinball_resid=0.15895 brier_exc=0.06027 logloss_exc=0.37511 roc_auc_exc=0.2616099590631501 pr_auc_exc=0.03682761526761681
[LOSO Quantile(log-resid)] fold=03 site=370408.9455206478_3750865.96396696 n_test=30121 pos_rate=0.0198 avg_pinball_resid=0.12188 brier_exc=0.01588 logloss_exc=0.06397 roc_auc_exc=0.9693136144927389 pr_auc_exc=0.4403489200026227
[LOSO Quantile(log-resid)] fold=04 site=373408.60980845685_3746402.1236518873 n_test=20070 pos_rate=0.0295 avg_pinball_resid=0.21474 brier_exc=0.19074 logloss_exc=0.58310 roc_auc_exc=0.6253610757739855 pr_auc_exc=0.19356642085483733
[LOSO Quantile(log-resid)] fold=05 site=376372.05412800366_3748307.72861234

In [125]:
df_q_resid_metrics_newtuning_wojulian.to_parquet(save_path +f"loso_quantile_metrics{file_suffix}.parquet", index=False)
df_q_resid_pred_newtuning_wojulian.to_parquet(save_path +f"loso_quantile_predictions{file_suffix}.parquet", index=False)

In [126]:
df_bin_metrics_newtuning_wojulian = pd.read_parquet(save_path +f"loso_binary_metrics{file_suffix}.parquet")
df_bin_pred_newtuning_wojulian    = pd.read_parquet(save_path +f"loso_binary_predictions{file_suffix}.parquet")

df_q_metrics_newtuning_wojulian   = pd.read_parquet(save_path +f"loso_quantile_metrics{file_suffix}.parquet")
df_q_pred_newtuning_wojulian      = pd.read_parquet(save_path +f"loso_quantile_predictions{file_suffix}.parquet")

In [127]:
count_in_alpha_intervals(df_q_pred_newtuning_wojulian, "pred_exceedance_probability", alphas)

,interval,count
0,"(-inf, 0.001]",0
1,"(0.001, 0.01]",342
2,"(0.01, 0.03]",1034
3,"(0.03, 0.05]",918
4,"(0.05, 0.075]",1647
5,"(0.075, 0.1]",1365
6,"(0.1, 0.2]",6158
7,"(0.2, 0.3]",2800
8,"(0.3, 0.4]",5061
9,"(0.4, 0.5]",8678


In [128]:
make_monitor_loso_summary(monitor_coord_map, df_bin_metrics_newtuning_wojulian, df_q_metrics_newtuning_wojulian,
                          metrics = ["baseline_brier","bin_brier", "bin_bss",
                                     "bin_roc_auc", "bin_pr_auc", "q_brier", 
                                     "q_bss", "q_roc_auc", "q_pr_auc", ])

,Monitor,n,n_exceed,prop_exceed,baseline_brier,baseline_brier,bin_brier,bin_bss,bin_roc_auc,bin_pr_auc,q_brier,q_bss,q_roc_auc,q_pr_auc
0,Chico,2521,2487,0.986513,0.013305,0.013305,0.454858,-33.187469,0.739421,0.995722,0.526449,-38.568326,0.783036,0.996502
1,ElSegundo,5279,312,0.059102,0.055609,0.055609,0.070537,-0.268441,0.303979,0.038771,0.060274,-0.083897,0.261610,0.036828
2,ElmAve,32216,2107,0.065402,0.061125,0.061125,0.090581,-0.481910,0.911469,0.538046,0.045813,0.250500,0.877734,0.471506
3,FirstMethodist,30759,1935,0.062908,0.058951,0.058951,0.045846,0.222301,0.940455,0.642452,0.037752,0.359608,0.939151,0.639532
4,GStreet,22217,1677,0.075483,0.069785,0.069785,0.105310,-0.509066,0.885424,0.554889,0.056272,0.193639,0.867870,0.544802
5,GuenserPark,27326,1279,0.046805,0.044615,0.044615,0.066175,-0.483260,0.929460,0.541094,0.030563,0.314957,0.943939,0.550929
6,HarborPark,26127,766,0.029318,0.028459,0.028459,0.093098,-2.271322,0.838342,0.289720,0.074347,-1.612430,0.845872,0.284147
7,Hudson,32091,3559,0.110903,0.098604,0.098604,0.082447,0.163853,0.941928,0.742752,0.054076,0.451582,0.939472,0.717979
8,InnerPort,29107,3905,0.134160,0.116161,0.116161,0.108458,0.066313,0.808204,0.533477,0.124398,-0.070908,0.773197,0.514309
9,Judson,30014,2467,0.082195,0.075439,0.075439,0.081076,-0.074727,0.966183,0.719071,0.043575,0.422383,0.962335,0.717031


For binary classifier:
- Model is useful for 6 sites (BSS > 0)
 
For QXGB:
- Model is useful for 7 sites


Same as above

In [129]:
exp_bin = expected_exceedances_by_group(df_bin_pred_newtuning_wojulian, group_col = "Monitor")
exp_q   = expected_exceedances_by_group(df_q_pred_newtuning_wojulian, group_col = "Monitor")
exp_bin.merge(exp_q, on = ['Monitor', 'n'], how = 'left', suffixes = ('_bin', '_q'))

,Monitor,n,observed_exceed_bin,expected_exceed_bin,observed_exceed_q,expected_exceed_q
0,Chico,2521,2487,1102,2487,854
1,ElSegundo,5279,312,197,312,71
2,ElmAve,32216,2107,6526,2107,2075
3,FirstMethodist,30759,1935,3622,1935,1372
4,GStreet,22217,1677,4958,1677,2432
5,GuenserPark,27326,1279,3785,1279,1918
6,HarborPark,26127,766,4637,766,3759
7,Hudson,32091,3559,7057,3559,3208
8,InnerPort,29107,3905,5910,3905,8215
9,Judson,30014,2467,6187,2467,3552


## 8020 tests

### Helpers

In [150]:
# -----------------------------
# 80/20 RANDOM SPLIT (row-wise)
# -----------------------------
def random8020_binary_log(df, predictors, bin_params, *, time_col, test_size=0.2, seed=42, verbose=True):
    """
    One random 80/20 split (row-wise)
    Returns (df_metrics, df_pred) similar to LOSO versions.
    """
    tr_idx, te_idx = train_test_split(np.arange(len(df)), test_size=test_size, random_state=seed, shuffle=True)

    train = df.iloc[tr_idx].copy()
    test  = df.iloc[te_idx].copy()

    Xtr = train[predictors]
    ytr = train["y_exc"].to_numpy()
    Xte = test[predictors]
    yte = test["y_exc"].to_numpy()

    # imbalance weight
    pos = max(int(ytr.sum()), 1)
    neg = max(int(len(ytr) - ytr.sum()), 1)
    scale_pos_weight = neg / pos

    clf = xgb.XGBClassifier(**bin_params, scale_pos_weight=scale_pos_weight)
    clf.fit(Xtr, ytr)
    p = clf.predict_proba(Xte)[:, 1]

    # per-row predictions table
    pred_df = test[[time_col, "mon_utm_x", "mon_utm_y", "Monitor"]].copy()
    pred_df["obs"] = test["y"].to_numpy()
    pred_df["pred_exceedance_probability"] = p
    pred_df["fold"] = 1
    pred_df["site"] = "random_80_20"

    # metrics (single "fold")
    row = dict(
        fold=1,
        site="random_80_20",
        n_test=len(yte),
        pos_rate_test=float(yte.mean()),
        brier=brier_score_loss(yte, p),
        logloss=log_loss(yte, p, labels=[0, 1]),
    )
    if len(np.unique(yte)) == 2:
        row["roc_auc"] = roc_auc_score(yte, p)
        row["pr_auc"]  = average_precision_score(yte, p)
    else:
        row["roc_auc"] = np.nan
        row["pr_auc"]  = np.nan

    if verbose:
        print(
            f"[Random 80/20 Binary(log-lag)] "
            f"n_test={row['n_test']} pos_rate={row['pos_rate_test']:.4f} "
            f"brier={row['brier']:.5f} logloss={row['logloss']:.5f} "
            f"roc_auc={row['roc_auc'] if not np.isnan(row['roc_auc']) else 'NA'} "
            f"pr_auc={row['pr_auc'] if not np.isnan(row['pr_auc']) else 'NA'}"
        )

    return pd.DataFrame([row]), pred_df


def random8020_quantile_residual_log(df, predictors, alphas, q_params, *, time_col,
                                     test_size=0.2, seed=42, verbose=True):
    """
    One random 80/20 split (row-wise)
    Returns (df_metrics, df_pred) similar to LOSO versions.
    """
    tr_idx, te_idx = train_test_split(np.arange(len(df)), test_size=test_size, random_state=seed, shuffle=True)

    train = df.iloc[tr_idx].copy()
    test  = df.iloc[te_idx].copy()

    Xtr = train[predictors]
    ytr = train["z"].to_numpy()
    Xte = test[predictors]
    yte_r   = test["z"].to_numpy()
    yte_exc = test["y_exc"].to_numpy()

    qreg = xgb.XGBRegressor(**q_params, quantile_alpha=alphas)
    qreg.fit(Xtr, ytr)

    Qz = _to_mat(qreg.predict(Xte), n=len(Xte), K=len(alphas))

    # pinball
    pinballs = [mean_pinball_loss(yte_r, Qz[:, j], alpha=alphas[j]) for j in range(len(alphas))]
    avg_pinball = float(np.mean(pinballs))

    # exceedance prob at log(30)
    p_exc = exceed_prob_from_quantiles(Qz, ps=alphas, threshold=THRESH_LOG)

    pred_df = test[[time_col, "mon_utm_x", "mon_utm_y", "Monitor"]].copy()
    pred_df["obs"] = test["y"].to_numpy()
    pred_df["pred_exceedance_probability"] = p_exc
    pred_df["fold"] = 1
    pred_df["site"] = "random_80_20"

    row = dict(
        fold=1,
        site="random_80_20",
        n_test=len(yte_exc),
        pos_rate_test=float(yte_exc.mean()),
        avg_pinball_resid=avg_pinball,
        brier_exc=brier_score_loss(yte_exc, p_exc),
        logloss_exc=log_loss(yte_exc, p_exc, labels=[0, 1]),
    )
    if len(np.unique(yte_exc)) == 2:
        row["roc_auc_exc"] = roc_auc_score(yte_exc, p_exc)
        row["pr_auc_exc"]  = average_precision_score(yte_exc, p_exc)
    else:
        row["roc_auc_exc"] = np.nan
        row["pr_auc_exc"]  = np.nan

    if verbose:
        print(
            f"[Random 80/20 Quantile(log-resid)] "
            f"n_test={row['n_test']} pos_rate={row['pos_rate_test']:.4f} "
            f"avg_pinball_resid={row['avg_pinball_resid']:.5f} "
            f"brier_exc={row['brier_exc']:.5f} logloss_exc={row['logloss_exc']:.5f} "
            f"roc_auc_exc={row['roc_auc_exc'] if not np.isnan(row['roc_auc_exc']) else 'NA'} "
            f"pr_auc_exc={row['pr_auc_exc'] if not np.isnan(row['pr_auc_exc']) else 'NA'}"
        )

    return pd.DataFrame([row]), pred_df

In [151]:
def make_8020_summary_df(
    bin_metrics_df: pd.DataFrame,
    q_metrics_df: pd.DataFrame,
    *,
    include_flags: bool = True,
    metrics: list | None = None,
):
    """
    Build a multi-row 80/20 summary from many seeds.
    Inputs must include column 'seed'.
    Returns a DataFrame with one row per seed plus a final Mean row (seed='Mean').

    Uses:
      r = pos_rate_test (from binary metrics)
      baseline_brier = r(1-r)
      BSS = 1 - (brier_model / baseline_brier)
    """

    # Merge per-seed (assumes same seeds appear in both)
    m = bin_metrics_df.merge(q_metrics_df, on="seed", suffixes=("_bin", "_q"), how="inner").copy()

    # Pull needed columns (robustly)
    r = m["pos_rate_test_bin"].astype(float)
    m["baseline_brier"] = r * (1.0 - r)

    m["bin_brier"]   = m["brier"].astype(float)
    m["bin_logloss"] = m["logloss"].astype(float)
    m["bin_roc_auc"] = m.get("roc_auc", np.nan)
    m["bin_pr_auc"]  = m.get("pr_auc", np.nan)

    m["q_avg_pinball_resid"] = m.get("avg_pinball_resid", np.nan)
    m["q_brier"]   = m["brier_exc"].astype(float)
    m["q_logloss"] = m["logloss_exc"].astype(float)
    m["q_roc_auc"] = m.get("roc_auc_exc", np.nan)
    m["q_pr_auc"]  = m.get("pr_auc_exc", np.nan)

    # n_test / pos_rate_test
    m["n_test"] = m["n_test_bin"]
    m["pos_rate_test"] = m["pos_rate_test_bin"]

    denom = m["baseline_brier"].to_numpy()
    denom_safe = np.where((denom > 0) & np.isfinite(denom), denom, np.nan)

    m["bin_bss"] = 1.0 - (m["bin_brier"].to_numpy() / denom_safe)
    m["q_bss"]   = 1.0 - (m["q_brier"].to_numpy() / denom_safe)

    if include_flags:
        m["bin_brier_better_than_baseline"] = m["bin_brier"] < m["baseline_brier"]
        m["q_brier_better_than_baseline"]   = m["q_brier"] < m["baseline_brier"]

    # Default column set (same as earlier, plus seed)
    default_cols = [
        "seed", "n_test", "pos_rate_test", "baseline_brier",
        "bin_brier", "bin_bss", "bin_logloss", "bin_roc_auc", "bin_pr_auc",
        "q_brier", "q_bss", "q_logloss", "q_roc_auc", "q_pr_auc",
        "q_avg_pinball_resid",
    ]
    if include_flags:
        default_cols += ["bin_brier_better_than_baseline", "q_brier_better_than_baseline"]

    if metrics is None:
        cols = default_cols
    else:
        base_cols = ["seed", "n_test", "pos_rate_test", "baseline_brier"]
        cols = base_cols + metrics
        cols = [c for c in cols if c in m.columns]

    out = m[cols].copy()

    # Add Mean row (numeric means; keep booleans as NA)
    mean_row = {"seed": "Mean"}
    for c in out.columns:
        if c == "seed":
            continue
        if pd.api.types.is_bool_dtype(out[c]):
            mean_row[c] = np.nan
        else:
            mean_row[c] = pd.to_numeric(out[c], errors="coerce").mean()

    out = pd.concat([out, pd.DataFrame([mean_row])], ignore_index=True)
    return out

### New tunings

In [180]:
seeds = list(range(1, 21))

In [181]:
# Run 8020 for each seed and store in df
bin_metrics_list = []
q_metrics_list = []
bin_pred_list = []
q_pred_list = []

for s in seeds:
    print("Running seed " + str(s))
    bin_metrics_8020, bin_pred_8020 = random8020_binary_log(
        df, hourly_predictors, bin_params_new, time_col=time_col, test_size=0.2, seed=s, verbose=False
    )
    q_metrics_8020, q_pred_8020 = random8020_quantile_residual_log(
        df, hourly_predictors, alphas, q_params_new, time_col=time_col, test_size=0.2, seed=s, verbose=False
    )

    # attach seed to the one-row metrics
    bm = bin_metrics_8020.copy()
    bm.insert(0, "seed", s)
    qm = q_metrics_8020.copy()
    qm.insert(0, "seed", s)

    bin_metrics_list.append(bm)
    q_metrics_list.append(qm)

    # keep predictions as list of dfs (also add seed column)
    bp = bin_pred_8020.copy()
    bp.insert(0, "seed", s)
    qp = q_pred_8020.copy()
    qp.insert(0, "seed", s)
    bin_pred_list.append(bp)
    q_pred_list.append(qp)

# combined metrics frames (20 rows each)
bin_metrics_8020_df = pd.concat(bin_metrics_list, ignore_index=True)
q_metrics_8020_df   = pd.concat(q_metrics_list, ignore_index=True)

# combined pred frames (20%*20 rows each)
bin_pred_8020_df = pd.concat(bin_pred_list, ignore_index=True)
q_pred_8020_df   = pd.concat(q_pred_list, ignore_index=True)

Running seed 1
Running seed 2
Running seed 3
Running seed 4
Running seed 5
Running seed 6
Running seed 7
Running seed 8
Running seed 9
Running seed 10
Running seed 11
Running seed 12
Running seed 13
Running seed 14
Running seed 15
Running seed 16
Running seed 17
Running seed 18
Running seed 19
Running seed 20


In [183]:
file_suffix = "_2ppb"

In [186]:
bin_metrics_8020_df.to_parquet(save_path +f"bin_metrics_8020_df{file_suffix}.parquet", index=False)
q_metrics_8020_df.to_parquet(save_path +f"q_metrics_8020_df{file_suffix}.parquet", index=False)

bin_pred_8020_df.to_parquet(save_path +f"bin_pred_8020_df{file_suffix}.parquet", index=False)
q_pred_8020_df.to_parquet(save_path +f"q_pred_8020_df{file_suffix}.parquet", index=False)

In [187]:
bin_metrics_8020_df_newtuning = pd.read_parquet(save_path +f"bin_metrics_8020_df{file_suffix}.parquet")
q_metrics_8020_df_newtuning   = pd.read_parquet(save_path +f"q_metrics_8020_df{file_suffix}.parquet")

bin_pred_8020_df_newtuning = pd.read_parquet(save_path +f"bin_pred_8020_df{file_suffix}.parquet")
q_pred_8020_df_newtuning   = pd.read_parquet(save_path +f"q_pred_8020_df{file_suffix}.parquet")

In [188]:
make_8020_summary_df(bin_metrics_8020_df_newtuning, q_metrics_8020_df_newtuning,
                     metrics = ["bin_brier", "bin_bss",
                                "bin_roc_auc", "bin_pr_auc", "q_brier", 
                                "q_bss", "q_roc_auc", "q_pr_auc"])


,seed,n_test,pos_rate_test,baseline_brier,bin_brier,bin_bss,bin_roc_auc,bin_pr_auc,q_brier,q_bss,q_roc_auc,q_pr_auc
0,1,73167.0,0.071863,0.066699,0.060788,0.088617,0.971718,0.791742,0.041349,0.380067,0.927043,0.632278
1,2,73167.0,0.072943,0.067622,0.061017,0.097674,0.972012,0.795123,0.041290,0.389393,0.929950,0.643426
2,3,73167.0,0.073708,0.068275,0.060937,0.107478,0.969626,0.791687,0.041917,0.386065,0.926774,0.642174
3,4,73167.0,0.073530,0.068124,0.060849,0.106784,0.971211,0.796203,0.041899,0.384955,0.929014,0.641603
4,5,73167.0,0.072997,0.067669,0.061220,0.095293,0.970374,0.792405,0.041594,0.385323,0.927927,0.638566
5,6,73167.0,0.074118,0.068625,0.060454,0.119058,0.970966,0.792539,0.042497,0.380730,0.926766,0.632755
6,7,73167.0,0.074214,0.068706,0.060152,0.124507,0.970899,0.795323,0.042404,0.382827,0.926840,0.637652
7,8,73167.0,0.073995,0.068520,0.061920,0.096319,0.971143,0.793125,0.042277,0.383000,0.927910,0.633331
8,9,73167.0,0.074050,0.068566,0.061161,0.108009,0.971050,0.799344,0.042279,0.383381,0.927720,0.640180
9,10,73167.0,0.073243,0.067879,0.061217,0.098149,0.971010,0.793077,0.041945,0.382065,0.926669,0.634859


In [190]:
exp_bin = expected_exceedances_by_group(bin_pred_8020_df_newtuning, group_col = "seed")
exp_q   = expected_exceedances_by_group(q_pred_8020_df_newtuning, group_col = "seed")
exp_bin.merge(exp_q, on = ['seed', 'n'], how = 'left', suffixes = ('_bin', '_q'))

,seed,n,observed_exceed_bin,expected_exceed_bin,observed_exceed_q,expected_exceed_q
0,1,73167,5258,12921,5258,5272
1,2,73167,5337,13009,5337,5302
2,3,73167,5393,12904,5393,5266
3,4,73167,5380,12980,5380,5291
4,5,73167,5341,12957,5341,5306
5,6,73167,5423,12948,5423,5352
6,7,73167,5430,12912,5430,5300
7,8,73167,5414,13130,5414,5400
8,9,73167,5418,13088,5418,5303
9,10,73167,5359,13006,5359,5333


## LOEO tests

### Helpers

In [192]:
# -----------------------------
# LOEO SPLIT 
# -----------------------------
def loeo_binary_log(df, predictors, bin_params, *, time_col, seed=42, verbose=True):
    df[time_col] = pd.to_datetime(df[time_col])
    tz = "America/Los_Angeles"
    is_test = df[time_col].between(pd.Timestamp("2021-07-01", tz=tz), pd.Timestamp("2021-08-31", tz=tz), inclusive="both")

    te_idx = np.flatnonzero(is_test.to_numpy())
    tr_idx = np.flatnonzero((~is_test).to_numpy())

    train = df.iloc[tr_idx].copy()
    test  = df.iloc[te_idx].copy()

    Xtr = train[predictors]
    ytr = train["y_exc"].to_numpy()
    Xte = test[predictors]
    yte = test["y_exc"].to_numpy()

    # imbalance weight
    pos = max(int(ytr.sum()), 1)
    neg = max(int(len(ytr) - ytr.sum()), 1)
    scale_pos_weight = neg / pos

    clf = xgb.XGBClassifier(**bin_params, scale_pos_weight=scale_pos_weight)
    clf.fit(Xtr, ytr)
    p = clf.predict_proba(Xte)[:, 1]

    # per-row predictions table
    pred_df = test[[time_col, "mon_utm_x", "mon_utm_y", "Monitor"]].copy()
    pred_df["obs"] = test["y"].to_numpy()
    pred_df["pred_exceedance_probability"] = p
    pred_df["fold"] = 1
    pred_df["site"] = "loeo"

    # metrics (single "fold")
    row = dict(
        fold=1,
        site="loeo",
        n_test=len(yte),
        pos_rate_test=float(yte.mean()),
        brier=brier_score_loss(yte, p),
        logloss=log_loss(yte, p, labels=[0, 1]),
    )
    if len(np.unique(yte)) == 2:
        row["roc_auc"] = roc_auc_score(yte, p)
        row["pr_auc"]  = average_precision_score(yte, p)
    else:
        row["roc_auc"] = np.nan
        row["pr_auc"]  = np.nan

    if verbose:
        print(
            f"[Leave-one-event-out Binary(log-lag)] "
            f"n_test={row['n_test']} pos_rate={row['pos_rate_test']:.4f} "
            f"brier={row['brier']:.5f} logloss={row['logloss']:.5f} "
            f"roc_auc={row['roc_auc'] if not np.isnan(row['roc_auc']) else 'NA'} "
            f"pr_auc={row['pr_auc'] if not np.isnan(row['pr_auc']) else 'NA'}"
        )

    return pd.DataFrame([row]), pred_df


def loeo_quantile_residual_log(df, predictors, alphas, q_params, *, time_col, seed=42, verbose=True):
    df[time_col] = pd.to_datetime(df[time_col])
    tz = "America/Los_Angeles"
    is_test = df[time_col].between(pd.Timestamp("2021-07-01", tz=tz), pd.Timestamp("2021-08-31", tz=tz), inclusive="both")

    te_idx = np.flatnonzero(is_test.to_numpy())
    tr_idx = np.flatnonzero((~is_test).to_numpy())

    train = df.iloc[tr_idx].copy()
    test  = df.iloc[te_idx].copy()

    Xtr = train[predictors]
    ytr = train["z"].to_numpy()
    Xte = test[predictors]
    yte_r   = test["z"].to_numpy()
    yte_exc = test["y_exc"].to_numpy()

    qreg = xgb.XGBRegressor(**q_params, quantile_alpha=alphas)
    qreg.fit(Xtr, ytr)

    Qz = _to_mat(qreg.predict(Xte), n=len(Xte), K=len(alphas))

    # pinball
    pinballs = [mean_pinball_loss(yte_r, Qz[:, j], alpha=alphas[j]) for j in range(len(alphas))]
    avg_pinball = float(np.mean(pinballs))

    # exceedance prob at log(30)
    p_exc = exceed_prob_from_quantiles(Qz, ps=alphas, threshold=THRESH_LOG)

    pred_df = test[[time_col, "mon_utm_x", "mon_utm_y", "Monitor"]].copy()
    pred_df["obs"] = test["y"].to_numpy()
    pred_df["pred_exceedance_probability"] = p_exc
    pred_df["fold"] = 1
    pred_df["site"] = "loeo"

    row = dict(
        fold=1,
        site="loeo",
        n_test=len(yte_exc),
        pos_rate_test=float(yte_exc.mean()),
        avg_pinball_resid=avg_pinball,
        brier_exc=brier_score_loss(yte_exc, p_exc),
        logloss_exc=log_loss(yte_exc, p_exc, labels=[0, 1]),
    )
    if len(np.unique(yte_exc)) == 2:
        row["roc_auc_exc"] = roc_auc_score(yte_exc, p_exc)
        row["pr_auc_exc"]  = average_precision_score(yte_exc, p_exc)
    else:
        row["roc_auc_exc"] = np.nan
        row["pr_auc_exc"]  = np.nan

    if verbose:
        print(
            f"[Leave-one-event-out Quantile(log-resid)] "
            f"n_test={row['n_test']} pos_rate={row['pos_rate_test']:.4f} "
            f"avg_pinball_resid={row['avg_pinball_resid']:.5f} "
            f"brier_exc={row['brier_exc']:.5f} logloss_exc={row['logloss_exc']:.5f} "
            f"roc_auc_exc={row['roc_auc_exc'] if not np.isnan(row['roc_auc_exc']) else 'NA'} "
            f"pr_auc_exc={row['pr_auc_exc'] if not np.isnan(row['pr_auc_exc']) else 'NA'}"
        )

    return pd.DataFrame([row]), pred_df

### New tunings

In [193]:
# Run LOEO
bin_metrics_loeo, bin_pred_loeo = loeo_binary_log(
    df, hourly_predictors, bin_params_new, time_col='day', verbose=False
)
q_metrics_loeo, q_pred_loeo = loeo_quantile_residual_log(
    df, hourly_predictors, alphas, q_params_new, time_col='day', verbose=False
)


In [194]:
q_metrics_loeo.insert(0, "seed", 42)
bin_metrics_loeo.insert(0, "seed", 42)
make_8020_summary_df(bin_metrics_loeo, q_metrics_loeo,
                     metrics = ["bin_brier", "bin_bss",
                                "bin_roc_auc", "bin_pr_auc", "q_brier", 
                                "q_bss", "q_roc_auc", "q_pr_auc"])

,seed,n_test,pos_rate_test,baseline_brier,bin_brier,bin_bss,bin_roc_auc,bin_pr_auc,q_brier,q_bss,q_roc_auc,q_pr_auc
0,42,17835.0,0.019064,0.0187,0.0324,-0.732583,0.860671,0.162067,0.017827,0.046675,0.82892,0.134406
1,Mean,17835.0,0.019064,0.0187,0.0324,-0.732583,0.860671,0.162067,0.017827,0.046675,0.82892,0.134406


In [195]:
exp_bin = expected_exceedances_by_group(bin_pred_loeo, group_col = "site")
exp_q   = expected_exceedances_by_group(q_pred_loeo, group_col = "site")
exp_bin.merge(exp_q, on = ['site', 'n'], how = 'left', suffixes = ('_bin', '_q'))

,site,n,observed_exceed_bin,expected_exceed_bin,observed_exceed_q,expected_exceed_q
0,loeo,17835,340,1426,340,352
1,Mean,17835,340,1426,340,352
2,Sum,17835,340,1426,340,352
